# Gold — Fact Monthly Economics (Kimball Periodic Snapshot)
Joins Silver tables and unpivots into a narrow periodic snapshot fact table following Kimball conventions.

**Sources:** `silver.exchange_rates`, `silver.central_bank_indicators`, `silver.gdp_indicators`  
**Dimensions:** `gold.dim_date`, `gold.dim_indicator`, `gold.dim_source`, `gold.dim_geography`  
**Grain:** One row per calendar month per economic indicator

In [ ]:
df_fx  = spark.read.table("silver.exchange_rates")
df_cb  = spark.read.table("silver.central_bank_indicators")
df_gdp = spark.read.table("silver.gdp_indicators")

# Load dim_indicator to facilitate the unpivot and FK lookup
if spark.catalog.tableExists("gold.dim_indicator"):
    df_ind = spark.read.table("gold.dim_indicator")
    df_ind.createOrReplaceTempView("v_dim_indicator")
else:
    print("Warning: gold.dim_indicator missing! Ensure dimensions are created first.")

df_fx.createOrReplaceTempView("v_exchange_rates")
df_cb.createOrReplaceTempView("v_central_bank_indicators")
df_gdp.createOrReplaceTempView("v_gdp_indicators")

In [ ]:
df_gold = spark.sql("""
    WITH fx_monthly AS (
        SELECT
            YEAR(date) AS year, MONTH(date) AS month,
            ROUND(AVG(iskusd_close), 6)  AS avg_iskusd,
            ROUND(AVG(eurusd_close), 6)  AS avg_eurusd,
            ROUND(AVG(iskeur_close), 6)  AS avg_iskeur,
            ROUND(AVG(omx_close), 4)     AS avg_omx
        FROM v_exchange_rates
        GROUP BY 1, 2
    ),
    policy AS (
        SELECT year, month, ROUND(value, 4) AS policy_rate
        FROM (
            SELECT 
                YEAR(date) AS year, MONTH(date) AS month, value,
                ROW_NUMBER() OVER (PARTITION BY YEAR(date), MONTH(date) ORDER BY date DESC) AS rn
            FROM v_central_bank_indicators
            WHERE metric = 'policy_rate'
        ) WHERE rn = 1
    ),
    cpi AS (
        SELECT YEAR(date) AS year, MONTH(date) AS month, ROUND(value, 4) AS cpi
        FROM v_central_bank_indicators WHERE metric = 'cpi'
    ),
    gdp AS (
        SELECT year, q, ROUND(value, 2) AS gdp_yoy_growth
        FROM v_gdp_indicators
    ),
    combined_staged AS (
        SELECT
            COALESCE(fx.year, p.year, c.year, g.year) AS year,
            COALESCE(fx.month, p.month, c.month, 
                CASE WHEN g.q=1 THEN 3 WHEN g.q=2 THEN 6 WHEN g.q=3 THEN 9 ELSE 12 END) AS month,
            fx.avg_eurusd, fx.avg_iskusd, fx.avg_iskeur, fx.avg_omx,
            p.policy_rate, c.cpi, g.gdp_yoy_growth
        FROM fx_monthly fx
        FULL OUTER JOIN policy p ON fx.year = p.year AND fx.month = p.month
        FULL OUTER JOIN cpi    c ON COALESCE(fx.year, p.year) = c.year AND COALESCE(fx.month, p.month) = c.month
        FULL OUTER JOIN gdp    g ON COALESCE(fx.year, p.year, c.year) = g.year
            AND CASE
                WHEN COALESCE(fx.month, p.month, c.month) IN (1,2,3) THEN 1
                WHEN COALESCE(fx.month, p.month, c.month) IN (4,5,6) THEN 2
                WHEN COALESCE(fx.month, p.month, c.month) IN (7,8,9) THEN 3
                ELSE 4
            END = g.q
    ),
    unpivoted AS (
        SELECT 
            (year * 100 + month) AS month_key,
            (year * 10000 + month * 100 + 1) AS date_key,
            stack(7,
                1, CAST(avg_iskusd AS DOUBLE),
                2, CAST(avg_iskeur AS DOUBLE),
                3, CAST(avg_eurusd AS DOUBLE),
                4, CAST(avg_omx AS DOUBLE),
                5, CAST(policy_rate AS DOUBLE),
                6, CAST(cpi AS DOUBLE),
                7, CAST(gdp_yoy_growth AS DOUBLE)
            ) AS (indicator_key, value)
        FROM combined_staged
    )
    SELECT
        u.month_key, u.date_key, u.indicator_key, i.source_key,
        1 AS geography_key, u.value, FALSE AS is_estimated,
        CURRENT_TIMESTAMP() AS row_created_at, CURRENT_TIMESTAMP() AS row_updated_at
    FROM unpivoted u
    JOIN gold.dim_indicator i ON u.indicator_key = i.indicator_key
    WHERE u.value IS NOT NULL
""")

In [ ]:
df_gold.createOrReplaceTempView("v_fact_monthly_economics")

if not spark.catalog.tableExists("gold.fact_monthly_economics"):
    df_gold.write.format("delta").saveAsTable("gold.fact_monthly_economics")
else:
    spark.sql("""
        MERGE INTO gold.fact_monthly_economics AS target
        USING v_fact_monthly_economics AS source
        ON target.month_key = source.month_key
        AND target.indicator_key = source.indicator_key
        WHEN MATCHED THEN UPDATE SET
            target.date_key      = source.date_key,
            target.source_key    = source.source_key,
            target.value         = source.value,
            target.row_updated_at = source.row_updated_at
        WHEN NOT MATCHED THEN INSERT (
            month_key, date_key, indicator_key, source_key, geography_key, 
            value, is_estimated, row_created_at, row_updated_at
        )
        VALUES (
            source.month_key, source.date_key, source.indicator_key, source.source_key, 
            source.geography_key, source.value, source.is_estimated, 
            source.row_created_at, source.row_updated_at
        )
    """)

print(f"Gold table updated. Total rows: {spark.table('gold.fact_monthly_economics').count()}")